In [ ]:
import pandas as pd
import os

# read data
os.makedirs("../data/raw", exist_ok = True)
# data_url = "tbd"
oct_2019 = pd.read_csv("../data/raw/2019-Oct_partial.csv")

print(oct_2019.shape)
print(oct_2019.info())
print(oct_2019.head())

(141936, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141936 entries, 0 to 141935
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     141936 non-null  object 
 1   event_type     141936 non-null  object 
 2   product_id     141936 non-null  int64  
 3   category_id    141936 non-null  int64  
 4   category_code  96159 non-null   object 
 5   brand          121688 non-null  object 
 6   price          141936 non-null  float64
 7   user_id        141936 non-null  int64  
 8   user_session   141936 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 9.7+ MB
None
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       view    44600062  2103807459595387724   
1  2019-10-01 00:00:00 UTC       view     3900821  2053013552326770905   
2  2019-10-01 00:00:01 UTC       view    17200506  2053013559792632471   
3  2019-10-01 00:00:

In [17]:
# transform event date to datetime and extract the date only
oct_2019['event_time'] = pd.to_datetime(oct_2019['event_time'], utc = True)

# check if there's any unreasonable price (negative price)
(oct_2019["price"] < 0).any()

# remove duplicates
oct_2019 = oct_2019.drop_duplicates()

In [18]:
# Since currernt data is event-level, we need to transform the data into various data level based on the analysis we want to do.

# session-level analysis table
oct_2019['purchase_revenue'] = oct_2019['price'] * (oct_2019['event_type'] == 'purchase')  

session_df = oct_2019.groupby(['user_id', 'user_session']).agg(
    session_start = ('event_time', 'min'),
    session_end = ('event_time', 'max'),
    total_events = ('event_type', 'count'),
    view_count = ('event_type', lambda x: (x == 'view').sum()),
    cart_count = ('event_type', lambda x: (x == 'cart').sum()),
    purchase_count = ('event_type', lambda x: (x == 'purchase').sum()),
    total_revenue = ('purchase_revenue', 'sum')
).reset_index()

# session action
session_df['has_view'] = (session_df['view_count'] > 0).astype(int)
session_df['has_cart'] = (session_df['cart_count'] > 0).astype(int)
session_df['has_purchase'] = (session_df['purchase_count'] > 0).astype(int)

# session duration
#session_df['session_duration_sec'] = (
#    session_df['session_end'] - session_df['session_start']
#).dt.total_seconds()

session_df[session_df['has_purchase'] > 0].head()

,user_id,user_session,session_start,session_end,total_events,view_count,cart_count,purchase_count,total_revenue,has_view,has_cart,has_purchase
44,457360398,b7360dbb-427a-428b-a4fb-1f8872884d4d,2019-10-01 04:05:12+00:00,2019-10-01 04:22:53+00:00,14,13,0,1,51.46,1,0,1
170,504429960,54243b1c-14aa-4c29-9ddd-6607865700a1,2019-10-01 03:41:28+00:00,2019-10-01 04:14:48+00:00,45,44,0,1,275.66,1,0,1
216,512365496,d5aa25d2-afb4-4644-8edc-d8004c03091d,2019-10-01 03:57:17+00:00,2019-10-01 03:58:32+00:00,4,3,0,1,1415.48,1,0,1
247,512371939,7737b8d9-9359-420e-8c91-2fa5b1c89279,2019-10-01 04:32:12+00:00,2019-10-01 04:34:23+00:00,2,1,0,1,36.55,1,0,1
254,512373792,8b6814cb-7c0e-4cc9-8224-3a75672affff,2019-10-01 02:45:13+00:00,2019-10-01 02:53:14+00:00,12,11,0,1,213.61,1,0,1


In [19]:
# user-level analysis table
user_df = (session_df.groupby('user_id').agg(
    total_sessions = ('user_session', 'nunique'),
    total_views = ('view_count', 'sum'),
    total_carts = ('cart_count', 'sum'),
    total_purchases = ('purchase_count', 'sum'),
    total_revenue = ('total_revenue', 'sum')
)).reset_index()

user_df[user_df['total_sessions'] > 1].head()

# adding if user has purchase with 1 = true, 0 = false
user_df['has_purchase'] = (user_df['total_purchases'] > 0).astype(int)

In [20]:
# funnel table
funnel = pd.DataFrame(
    {
        'stage' : ['view', 'cart', 'purchase'],
        'sessions' : [
            session_df['has_view'].sum(),  # counts of sessions which has view
            session_df['has_cart'].sum(),  # counts of sessions which has cart
            session_df['has_purchase'].sum()  # counts of purchase which has purchase
        ]
    }
)

print(funnel)
# the ['sessions'] here is also funnel volume

      stage  sessions
0      view     34368
1      cart      1243
2  purchase      2133


In [21]:
# export analysis tables

# os.makedirs("../data/processed", exist_ok=True)

# df.to_csv("../data/analysis_table/events_cleaned.csv", index=False)
session_df.to_csv("../data/analysis_table/session_table.csv")
user_df.to_csv("../data/analysis_table/user_table.csv")
funnel.to_csv("../data/analysis_table/funnel_table.csv")